# Crawling Artikel Berita Detik.com (Sport & Finance)

**Tujuan:** mengumpulkan 200 artikel berita dari detik.com dengan dua label kategori, yaitu `sport` dan `finance` (masing-masing 100 artikel).

**Alur kerja:**
1. Instalasi dan import library
2. Konfigurasi sumber halaman indeks dan target jumlah artikel
3. Pengumpulan URL artikel dari halaman indeks
4. Ekstraksi isi berita menggunakan **trafilatura**
5. Penyimpanan hasil ke file CSV dengan 4 kolom: `id`, `isi_berita`, `label`, `url`
6. Load ulang file CSV dan menampilkan hasilnya

**Kenapa trafilatura?** Library ini dirancang khusus untuk *boilerplate removal*, yaitu memisahkan teks utama artikel dari elemen pengganggu seperti menu, iklan, komentar, dan footer. Jadi kita tidak perlu menulis selector HTML manual untuk badan artikel.

## 1. Instalasi dan Import Library

Library yang dipakai:
- `trafilatura` untuk mengambil halaman dan mengekstrak teks bersih dari artikel
- `requests` untuk mengambil HTML halaman indeks
- `beautifulsoup4` untuk mengurai halaman indeks dan mengumpulkan link artikel
- `pandas` untuk menyusun data tabular dan menyimpannya ke CSV

Jalankan cell ini sekali saja. Jika library sudah terpasang, pip akan melewatinya.

In [11]:
%pip install trafilatura requests beautifulsoup4 pandas -q

import re
import time
import random
import requests
import pandas as pd
import trafilatura
from bs4 import BeautifulSoup

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Konfigurasi Crawling

Beberapa hal yang diatur di sini:

- `KATEGORI` berisi label beserta URL halaman indeks detik.com. Halaman indeks dipilih karena menampilkan daftar berita terbaru secara berurutan dan mendukung paginasi lewat parameter `?page=`.
- `TARGET_PER_LABEL` diisi 100 supaya total keseluruhan menjadi 200 artikel.
- `HEADERS` berisi User-Agent agar request dikenali sebagai browser biasa, bukan bot mentah.
- `DELAY` adalah jeda antar request. Ini penting sebagai etika crawling supaya tidak membebani server detik.com.
- `POLA_ARTIKEL` adalah regex untuk mengenali URL artikel detik. Ciri khasnya adalah adanya segmen `/d-<angka>/` pada URL.

In [27]:
KATEGORI = {
    "sport":   "https://sport.detik.com/indeks",
    "finance": "https://finance.detik.com/indeks",
}

TARGET_PER_LABEL = 100      # 100 sport + 100 finance = 200 artikel
MAX_HALAMAN = 30       # batas aman paginasi indeks
DELAY = (1.0, 2.0)  # jeda acak antar request (detik)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
}

POLA_ARTIKEL = re.compile(r"^https://(sport|finance)\.detik\.com/[^/]+/d-\d+/")


def jeda():
    time.sleep(random.uniform(*DELAY))


print("Konfigurasi siap. Target total:",
      TARGET_PER_LABEL * len(KATEGORI), "artikel")

Konfigurasi siap. Target total: 200 artikel


## 3. Fungsi Pengumpulan URL Artikel

Tahap ini adalah *link harvesting*. Alurnya:

1. Request halaman indeks `https://sport.detik.com/indeks?page=N`
2. Parse HTML dengan BeautifulSoup dan ambil seluruh tag `<a href=...>`
3. Saring href yang cocok dengan pola URL artikel detik
4. Bersihkan query string (misal `?_ga=...`) supaya URL seragam
5. Simpan ke dalam `set` untuk menghindari duplikat, lalu ulangi ke halaman berikutnya sampai jumlah link mencukupi

Fungsi ini hanya mengumpulkan URL, belum mengambil isi berita. Pemisahan tahap seperti ini membuat proses lebih mudah di-debug.

In [28]:
def bersihkan_url(url: str) -> str:
    """Buang query string dan fragment agar URL seragam."""
    return url.split("?")[0].split("#")[0]


def ambil_link_indeks(url_indeks: str, jumlah: int, max_halaman: int = MAX_HALAMAN):
    """Kumpulkan URL artikel dari halaman indeks detik.com."""
    kumpulan = []
    sudah_ada = set()

    for halaman in range(1, max_halaman + 1):
        if len(kumpulan) >= jumlah:
            break

        url = f"{url_indeks}?page={halaman}"
        try:
            resp = requests.get(url, headers=HEADERS, timeout=15)
            resp.raise_for_status()
        except Exception as e:
            print(f"  [!] Gagal memuat {url} -> {e}")
            continue

        soup = BeautifulSoup(resp.text, "html.parser")
        baru = 0

        for a in soup.find_all("a", href=True):
            link = bersihkan_url(a["href"])
            if POLA_ARTIKEL.match(link) and link not in sudah_ada:
                sudah_ada.add(link)
                kumpulan.append(link)
                baru += 1
                if len(kumpulan) >= jumlah:
                    break

        print(
            f"  Halaman {halaman:>2}: +{baru} link (total {len(kumpulan)}/{jumlah})")
        jeda()

    return kumpulan


# Uji coba cepat pada 1 halaman
contoh = ambil_link_indeks(KATEGORI["sport"], jumlah=5, max_halaman=1)
contoh

  Halaman  1: +5 link (total 5/5)


['https://sport.detik.com/moto-gp/d-8666386/marc-marquez-disebut-di-puncak-karier-bahasan-cedera-tak-lagi-relevan',
 'https://sport.detik.com/fotosport/d-8666341/melihat-dari-dekat-latihan-para-atlet-atletik-di-mimika-sport-complex',
 'https://sport.detik.com/moto-gp/d-8666308/ai-ogura-siap-comeback-di-motogp-austria-enggan-pasang-target-tinggi',
 'https://sport.detik.com/g-sport/d-8666151/ketum-koni-tutup-program-sport-diplomacy-indonesia-timor-leste',
 'https://sport.detik.com/sport-lain/d-8666068/usai-dilantik-koni-ini-rencana-pengurus-pusat-kickboxing-indonesia']

## 4. Fungsi Ekstraksi Isi Berita dengan Trafilatura

Dua fungsi inti trafilatura yang dipakai:

- `trafilatura.fetch_url(url)` mengunduh halaman dan mengembalikan HTML mentah
- `trafilatura.extract(html, ...)` mengekstrak teks utama artikel

Parameter penting pada `extract`:
- `include_comments=False` untuk membuang kolom komentar pembaca
- `include_tables=False` untuk membuang tabel yang biasanya bukan narasi berita

Setelah teks didapat, dilakukan pembersihan ringan:
- Baris kosong berlebih dipadatkan menjadi satu spasi
- Artikel dengan panjang kurang dari 200 karakter dibuang karena kemungkinan besar berupa halaman foto, video, atau ekstraksi yang gagal

In [14]:
PANJANG_MINIMAL = 200  # karakter


def bersihkan_teks(teks: str) -> str:
    """Rapikan whitespace agar aman disimpan dalam satu sel CSV."""
    teks = re.sub(r"\s+", " ", teks)
    return teks.strip()


def ekstrak_artikel(url: str):
    """Ambil isi berita dari satu URL. Return None jika gagal / terlalu pendek."""
    try:
        html = trafilatura.fetch_url(url)
        if html is None:
            return None

        # Coba pakai favor_precision kalau versi trafilatura mendukung,
        # kalau tidak, fallback ke pemanggilan tanpa parameter tersebut.
        try:
            teks = trafilatura.extract(
                html,
                include_comments=False,
                include_tables=False,
                favor_precision=True,
            )
        except TypeError:
            teks = trafilatura.extract(
                html,
                include_comments=False,
                include_tables=False,
            )

        if not teks:
            return None

        teks = bersihkan_teks(teks)
        if len(teks) < PANJANG_MINIMAL:
            return None

        return teks
    except Exception:
        return None


# Uji coba pada satu artikel
uji = ekstrak_artikel(contoh[0])
print(uji[:400] if uji else "Ekstraksi gagal")

Marc Marquez Disebut di Puncak Karier, Bahasan Cedera Tak Lagi Relevan Marc Marquez seperti menjawab keraguan soal kondisi fisiknya dengan hasil di lintasan. Rider Ducati itu menyapu bersih MotoGP San Marino 2026 dengan meraih 37 poin dari maksimal 37 poin. Ia pun disebut sedang berada di puncak karier. Marc memenangi Sprint dan balapan utama di Misano. Tambahan tersebut membuat koleksi poinnya me


## 5. Proses Crawling Utama

Cell ini menggabungkan seluruh tahap sebelumnya:

1. Untuk setiap label (`sport` dan `finance`), kumpulkan kandidat URL lebih banyak dari target. Ini dilakukan karena sebagian URL pasti gagal diekstrak, misalnya berupa halaman video atau infografis.
2. Iterasi setiap URL, ekstrak isinya dengan trafilatura
3. Artikel yang berhasil disimpan ke dalam list `data` beserta labelnya
4. Berhenti ketika jumlah artikel per label sudah mencapai 100
5. Terakhir, `id` diberikan secara berurutan mulai dari 1 untuk seluruh dataset gabungan

Proses ini memerlukan waktu beberapa menit karena ada jeda antar request.

In [15]:
data = []

for label, url_indeks in KATEGORI.items():
    print(f"\n=== Crawling label: {label.upper()} ===")

    # Ambil kandidat 2x target untuk mengantisipasi kegagalan ekstraksi
    print("[1] Mengumpulkan URL artikel...")
    kandidat = ambil_link_indeks(url_indeks, jumlah=TARGET_PER_LABEL * 2)
    print(f"    Total kandidat URL: {len(kandidat)}")

    print("[2] Mengekstrak isi berita...")
    berhasil = 0
    for i, url in enumerate(kandidat, start=1):
        if berhasil >= TARGET_PER_LABEL:
            break

        isi = ekstrak_artikel(url)
        if isi:
            data.append({"isi_berita": isi, "label": label, "url": url})
            berhasil += 1
            if berhasil % 10 == 0:
                print(f"    {berhasil}/{TARGET_PER_LABEL} artikel berhasil")
        jeda()

    print(f"    Selesai. Label '{label}': {berhasil} artikel")

print(f"\nTOTAL ARTIKEL TERKUMPUL: {len(data)}")


=== Crawling label: SPORT ===
[1] Mengumpulkan URL artikel...
  Halaman  1: +18 link (total 18/200)
  Halaman  2: +19 link (total 37/200)
  Halaman  3: +20 link (total 57/200)
  Halaman  4: +18 link (total 75/200)
  Halaman  5: +20 link (total 95/200)
  Halaman  6: +20 link (total 115/200)
  Halaman  7: +20 link (total 135/200)
  Halaman  8: +20 link (total 155/200)
  Halaman  9: +20 link (total 175/200)
  Halaman 10: +19 link (total 194/200)
  Halaman 11: +6 link (total 200/200)
    Total kandidat URL: 200
[2] Mengekstrak isi berita...
    10/100 artikel berhasil
    20/100 artikel berhasil
    30/100 artikel berhasil
    40/100 artikel berhasil
    50/100 artikel berhasil
    60/100 artikel berhasil
    70/100 artikel berhasil
    80/100 artikel berhasil
    90/100 artikel berhasil
    100/100 artikel berhasil
    Selesai. Label 'sport': 100 artikel

=== Crawling label: FINANCE ===
[1] Mengumpulkan URL artikel...
  Halaman  1: +19 link (total 19/200)
  Halaman  2: +17 link (total 36

## 6. Menyimpan Hasil ke File CSV

List `data` diubah menjadi DataFrame pandas, lalu:
- Kolom `id` ditambahkan sebagai nomor urut mulai dari 1
- Urutan kolom disusun menjadi `id`, `isi_berita`, `label`, `url` sesuai ketentuan tugas
- Disimpan dengan `encoding="utf-8-sig"` supaya karakter Indonesia tetap terbaca benar saat file dibuka di Excel
- `index=False` supaya pandas tidak menambah kolom indeks tambahan

In [16]:
NAMA_FILE = "dataset_berita_detik.csv"

df = pd.DataFrame(data)
df.insert(0, "id", range(1, len(df) + 1))
df = df[["id", "isi_berita", "label", "url"]]

df.to_csv(NAMA_FILE, index=False, encoding="utf-8-sig")

print(f"Berhasil disimpan ke '{NAMA_FILE}'")
print("Jumlah baris :", len(df))
print("Kolom        :", list(df.columns))

Berhasil disimpan ke 'dataset_berita_detik.csv'
Jumlah baris : 200
Kolom        : ['id', 'isi_berita', 'label', 'url']


## 7. Load Ulang dan Menampilkan Hasil

Tahap verifikasi. File CSV dibaca ulang dari disk untuk memastikan datanya benar-benar tersimpan dengan baik, lalu ditampilkan:

- `head()` untuk melihat sampel baris awal
- `info()` untuk melihat tipe data dan jumlah nilai non-null
- `value_counts()` untuk memastikan distribusi label seimbang antara sport dan finance
- Statistik panjang teks untuk mengecek kualitas hasil ekstraksi

In [17]:
df_load = pd.read_csv(NAMA_FILE)

print("=" * 60)
print("DIMENSI DATA:", df_load.shape)
print("=" * 60)
df_load.head(10)

DIMENSI DATA: (200, 4)


,id,isi_berita,label,url
0,1,"Marc Marquez Disebut di Puncak Karier, Bahasan...",sport,https://sport.detik.com/moto-gp/d-8666386/marc...
1,2,Foto Sport Melihat dari Dekat Latihan Para Atl...,sport,https://sport.detik.com/fotosport/d-8666341/me...
2,3,"Ai Ogura Siap Comeback di MotoGP Austria, Engg...",sport,https://sport.detik.com/moto-gp/d-8666308/ai-o...
3,4,Ketum KONI Tutup Program Sport Diplomacy Indon...,sport,https://sport.detik.com/g-sport/d-8666151/ketu...
4,5,"Usai Dilantik KONI, Ini Rencana Pengurus Pusat...",sport,https://sport.detik.com/sport-lain/d-8666068/u...
5,6,"Lina Hisage, Atlet Muda Potensial Cabor Tolak ...",sport,https://sport.detik.com/sport-lain/d-8666076/l...
6,7,Tim Equestrian Indonesia Turunkan 12 Atlet di ...,sport,https://sport.detik.com/sport-lain/d-8665977/t...
7,8,Foto Sport Serunya 549 Siswa di Ciamis Adu Ket...,sport,https://sport.detik.com/fotosport/d-8665222/se...
8,9,'Marc Marquez adalah Makhluk Mars' Dalam dua s...,sport,https://sport.detik.com/moto-gp/d-8665638/marc...
9,10,"Alwi, Ubed, hingga Yusuf Petik Pengalaman dari...",sport,https://sport.detik.com/raket/d-8665674/alwi-u...


In [18]:
print("--- INFO DATAFRAME ---")
df_load.info()

print("\n--- DISTRIBUSI LABEL ---")
print(df_load["label"].value_counts())

print("\n--- CEK DUPLIKAT & MISSING VALUE ---")
print("URL duplikat  :", df_load["url"].duplicated().sum())
print("Missing value :\n", df_load.isnull().sum())

print("\n--- STATISTIK PANJANG ISI BERITA (karakter) ---")
panjang = df_load["isi_berita"].str.len()
print(panjang.describe())

--- INFO DATAFRAME ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          200 non-null    int64 
 1   isi_berita  200 non-null    object
 2   label       200 non-null    object
 3   url         200 non-null    object
dtypes: int64(1), object(3)
memory usage: 6.4+ KB

--- DISTRIBUSI LABEL ---
label
sport      100
finance    100
Name: count, dtype: int64

--- CEK DUPLIKAT & MISSING VALUE ---
URL duplikat  : 0
Missing value :
 id            0
isi_berita    0
label         0
url           0
dtype: int64

--- STATISTIK PANJANG ISI BERITA (karakter) ---
count     200.000000
mean     2446.470000
std      1280.072524
min       224.000000
25%      1791.750000
50%      2150.500000
75%      2917.750000
max      7149.000000
Name: isi_berita, dtype: float64


In [19]:
# Tampilkan contoh satu artikel dari tiap label
for label in df_load["label"].unique():
    sampel = df_load[df_load["label"] == label].iloc[0]
    print("=" * 70)
    print(f"ID    : {sampel['id']}")
    print(f"LABEL : {sampel['label']}")
    print(f"URL   : {sampel['url']}")
    print("-" * 70)
    print(sampel["isi_berita"][:600], "...")
    print()

ID    : 1
LABEL : sport
URL   : https://sport.detik.com/moto-gp/d-8666386/marc-marquez-disebut-di-puncak-karier-bahasan-cedera-tak-lagi-relevan
----------------------------------------------------------------------
Marc Marquez Disebut di Puncak Karier, Bahasan Cedera Tak Lagi Relevan Marc Marquez seperti menjawab keraguan soal kondisi fisiknya dengan hasil di lintasan. Rider Ducati itu menyapu bersih MotoGP San Marino 2026 dengan meraih 37 poin dari maksimal 37 poin. Ia pun disebut sedang berada di puncak karier. Marc memenangi Sprint dan balapan utama di Misano. Tambahan tersebut membuat koleksi poinnya menjadi 274, sama dengan Jorge Martin. Namun, Marc berhak memimpin klasemen karena unggul jumlah kemenangan balapan utama, 5-1. Performa itu sekaligus menegaskan kebangkitan Marc. Tujuh seri sebelumnya, ...

ID    : 101
LABEL : finance
URL   : https://finance.detik.com/berita-ekonomi-bisnis/d-8666470/penempatan-manajer-kopdes-ditargetkan-paling-lambat-25-september
--------------------